# 计算最终的结果

In [ ]:
import json
import pandas as pd
from typing import Dict, List
import os

def load_jsonl_results(jsonl_path: str) -> pd.DataFrame:
    """
    Load evaluation results from JSONL file.
    
    Args:
        jsonl_path: Path to the JSONL file containing evaluation results
    
    Returns:
        DataFrame with columns: question_id, correctness_reasoning, 
        error_diagnosis, suggestion_usefulness, clarity, hallucination, overall, notes
    """
    records = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                record = {
                    "question_id": item["question_id"],
                    "correctness_reasoning": item["scores"]["correctness_reasoning"],
                    "error_diagnosis": item["scores"]["error_diagnosis"],
                    "suggestion_usefulness": item["scores"]["suggestion_usefulness"],
                    "clarity": item["scores"]["clarity"],
                    "hallucination": item["scores"]["hallucination"],
                    "overall": item["scores"]["overall"],
                    "notes": item.get("notes", "")
                }
                records.append(record)
    
    df = pd.DataFrame(records)
    print(f"[OK] Loaded {len(df)} evaluation results from {jsonl_path}")
    return df


def comprehensive_quality_metrics(df: pd.DataFrame, 
                                   weights: Dict[str, float] = None) -> Dict:
    """
    Compute comprehensive quality metrics from evaluation results.
    
    Args:
        df: DataFrame with evaluation scores
        weights: Optional custom weights for dimensions
    
    Returns:
        Dictionary containing all quality metrics
    """
    dimensions = ["correctness_reasoning", "error_diagnosis", 
                  "suggestion_usefulness", "clarity"]
    
    # Default weights (可以根据你的论文调整)
    if weights is None:
        weights = {
            "correctness_reasoning": 0.30,
            "error_diagnosis": 0.30,
            "suggestion_usefulness": 0.20,
            "clarity": 0.20
        }
    
    # Validate weights sum to 1
    assert abs(sum(weights.values()) - 1.0) < 1e-6, "Weights must sum to 1.0"
    
    # ===== 1. Dimension Statistics =====
    weighted_mean = 0
    weighted_cv = 0
    dim_stats = {}
    
    for dim in dimensions:
        mean_val = df[dim].mean()
        std_val = df[dim].std()
        cv = std_val / mean_val if mean_val > 0 else 0
        
        dim_stats[dim] = {
            "mean": round(mean_val, 3),
            "std": round(std_val, 3),
            "cv": round(cv, 3),
            "min": float(df[dim].min()),
            "max": float(df[dim].max())
        }
        
        weighted_mean += weights[dim] * mean_val
        weighted_cv += weights[dim] * cv
    
    # ===== 2. Quality Score (Mean - Penalty × CV) =====
    alpha = 0.5  # Penalty coefficient for instability
    quality_score = weighted_mean - alpha * weighted_cv
    
    # ===== 3. Acceptance Metrics =====
    accepted = df[df['overall'] == 'Accept']
    rejected = df[df['overall'] == 'Reject']
    
    acceptance_rate = len(accepted) / len(df) if len(df) > 0 else 0
    rejection_rate = len(rejected) / len(df) if len(df) > 0 else 0
    
    # Mean quality of accepted items only
    if len(accepted) > 0:
        accepted_mean = accepted[dimensions].mean(axis=1).mean()
    else:
        accepted_mean = 0
    
    # ===== 4. Composite Score (Acceptance × Quality) =====
    composite_score = acceptance_rate * accepted_mean
    
    # ===== 5. Harmonic Quality Score (严格版本) =====
    harmonic_sum = sum(weights[dim] / df[dim].mean() 
                       for dim in dimensions if df[dim].mean() > 0)
    harmonic_qs = 1 / harmonic_sum if harmonic_sum > 0 else 0
    
    # ===== 6. Hallucination Statistics =====
    hallucination_count = (df['hallucination'] == 'Yes').sum()
    hallucination_rate = hallucination_count / len(df) if len(df) > 0 else 0
    
    return {
        # === Core Metrics ===
        "quality_score": round(quality_score, 3),
        "composite_score": round(composite_score, 3),
        "harmonic_quality_score": round(harmonic_qs, 3),
        
        # === Aggregated Statistics ===
        "weighted_mean": round(weighted_mean, 3),
        "weighted_cv": round(weighted_cv, 3),
        
        # === Acceptance Metrics ===
        "acceptance_rate": round(acceptance_rate, 3),
        "rejection_rate": round(rejection_rate, 3),
        "accepted_mean_quality": round(accepted_mean, 3),
        "total_items": len(df),
        "accepted_count": len(accepted),
        "rejected_count": len(rejected),
        
        # === Hallucination ===
        "hallucination_count": int(hallucination_count),
        "hallucination_rate": round(hallucination_rate, 3),
        
        # === Dimension Details ===
        "dimension_stats": dim_stats,
        
        # === Weights Used ===
        "weights": weights
    }


def print_metrics_report(metrics: Dict):
    """
    Pretty print the quality metrics in a readable format.
    """
    print("\n" + "="*70)
    print(" COMPREHENSIVE QUALITY METRICS REPORT ".center(70))
    print("="*70 + "\n")
    
    # Core Metrics
    print("📊 CORE METRICS")
    print("-" * 70)
    print(f"  Quality Score (Mean - α×CV):        {metrics['quality_score']:.3f} / 5.0")
    print(f"  Composite Score (Accept% × Quality): {metrics['composite_score']:.3f} / 5.0")
    print(f"  Harmonic Quality Score:             {metrics['harmonic_quality_score']:.3f} / 5.0")
    print()
    
    # Aggregated Stats
    print("📈 AGGREGATED STATISTICS")
    print("-" * 70)
    print(f"  Weighted Mean Score:                {metrics['weighted_mean']:.3f}")
    print(f"  Weighted Coefficient of Variation:  {metrics['weighted_cv']:.3f}")
    print()
    
    # Acceptance Metrics
    print("✅ ACCEPTANCE METRICS")
    print("-" * 70)
    print(f"  Total Items Evaluated:              {metrics['total_items']}")
    print(f"  Accepted Count:                     {metrics['accepted_count']} ({metrics['acceptance_rate']*100:.1f}%)")
    print(f"  Rejected Count:                     {metrics['rejected_count']} ({metrics['rejection_rate']*100:.1f}%)")
    print(f"  Mean Quality (Accepted Only):       {metrics['accepted_mean_quality']:.3f}")
    print()
    
    # Hallucination
    print("🚨 HALLUCINATION DETECTION")
    print("-" * 70)
    print(f"  Hallucination Cases:                {metrics['hallucination_count']} ({metrics['hallucination_rate']*100:.1f}%)")
    print()
    
    # Dimension Breakdown
    print("📋 DIMENSION-WISE BREAKDOWN")
    print("-" * 70)
    print(f"  {'Dimension':<25} {'Weight':<8} {'Mean':<8} {'Std':<8} {'CV':<8} {'Range':<12}")
    print("  " + "-" * 68)
    
    for dim, stats in metrics['dimension_stats'].items():
        weight = metrics['weights'][dim]
        dim_display = dim.replace('_', ' ').title()
        range_str = f"[{stats['min']:.1f}, {stats['max']:.1f}]"
        print(f"  {dim_display:<25} {weight:<8.2f} {stats['mean']:<8.3f} "
              f"{stats['std']:<8.3f} {stats['cv']:<8.3f} {range_str:<12}")
    
    print("\n" + "="*70 + "\n")


def save_metrics_to_json(metrics: Dict, output_path: str):
    """
    Save metrics to JSON file for later use.
    """
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print(f"[OK] Metrics saved to {output_path}")


def main():
    """
    Main function to compute quality metrics from JSONL evaluation results.
    """
    # ===== Configuration =====
    jsonl_path = "./judge_for_generate/eval_outputs_manual_review/zh/7/results.jsonl"  # 修改为你的文件路径
    output_json = "./judge_for_generate/eval_outputs_manual_review/quality_metrics_grade7.json"
    
    # Optional: Custom weights (如果不指定，使用默认权重)
    custom_weights = {
        "correctness_reasoning": 0.30,
        "error_diagnosis": 0.30,
        "suggestion_usefulness": 0.20,
        "clarity": 0.20
    }
    
    # ===== Load Data =====
    if not os.path.exists(jsonl_path):
        print(f"[ERROR] File not found: {jsonl_path}")
        return
    
    df = load_jsonl_results(jsonl_path)
    
    # ===== Compute Metrics =====
    metrics = comprehensive_quality_metrics(df, weights=custom_weights)
    
    # ===== Display Report =====
    print_metrics_report(metrics)
    
    # ===== Save Results =====
    save_metrics_to_json(metrics, output_json)
    
    # ===== Optional: Generate LaTeX Table =====
    print("\n📄 LaTeX Table (copy to your paper):")
    print("-" * 70)
    print(generate_latex_table(metrics))


def generate_latex_table(metrics: Dict) -> str:
    """
    Generate LaTeX table for the paper.
    """
    latex = r"""
\begin{table}[htbp]
\centering
\caption{Quality Metrics of Generated Pedagogical Explanations}
\label{tab:quality_metrics}
\begin{tabular}{lcc}
\toprule
\textbf{Metric} & \textbf{Value} & \textbf{Interpretation} \\
\midrule
Quality Score & %.3f & High quality + low variability \\
Weighted Mean & %.3f & Average across dimensions \\
Weighted CV & %.3f & Stability measure \\
Acceptance Rate & %.1f\%% & Passed quality assurance \\
Composite Score & %.3f & Efficiency × Quality \\
Harmonic Quality Score & %.3f & No weak dimension \\
\midrule
\multicolumn{3}{l}{\textit{Dimension Breakdown:}} \\
\midrule
""" % (
        metrics['quality_score'],
        metrics['weighted_mean'],
        metrics['weighted_cv'],
        metrics['acceptance_rate'] * 100,
        metrics['composite_score'],
        metrics['harmonic_quality_score']
    )
    
    for dim, stats in metrics['dimension_stats'].items():
        dim_display = dim.replace('_', ' ').title()
        latex += f"  {dim_display} & {stats['mean']:.2f} $\\pm$ {stats['std']:.2f} & CV={stats['cv']:.2f} \\\\\n"
    
    latex += r"""\bottomrule
\end{tabular}
\end{table}
"""
    return latex


if __name__ == "__main__":
    main()
